# Relire les traces et les essais en lot

Chaque question posée (`rag-ask`, `rag-batch`, notebooks) laisse une fiche de
décision dans `data/traces/traces.jsonl`. Ce cahier les rassemble en tableaux
pour repérer les refus, les citations douteuses et les coûts.

Prérequis : `uv sync --group notebooks`, puis le noyau `.venv`.

In [ ]:
import os
from pathlib import Path

# Travailler depuis la racine du dépôt (data/, .env), où que VS Code ouvre le cahier.
if Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd

from rag_rgpd.explain.render import format_trace
from rag_rgpd.explain.trace import find_trace, read_traces

## 1. Toutes les questions posées

In [ ]:
traces = read_traces()
df = pd.DataFrame([
    {"id": t.trace_id, "date": t.at, "question": t.question, "décision": t.decision,
     "code refus": t.refusal_code, "meilleure sim.": t.best_similarity,
     "k": t.params.get("k"), "seuil": t.params.get("min_similarity"),
     "modèle": t.llm_model, "citations": len(t.citations),
     "citations ok": all(c["verified"] for c in t.citations),
     "coût $": t.cost_usd, "cache": t.from_cache, "durée s": t.timings.get("total")}
    for t in traces
])
print(f"{len(df)} traces, coût total estimé {df['coût $'].fillna(0).sum():.4f} $" if len(df) else "aucune trace")
df.tail(20)

## 2. Décisions et refus

Le code de refus dit quelle barrière a décidé : `aucun_passage_pertinent`
(barrière 1, avant le LLM), `passages_insuffisants` (le modèle a dit « je ne
sais pas »), `citation_absente` ou `citation_non_verifiee` (contrôle final).

In [ ]:
if len(df):
    display(df.groupby(["décision", "code refus"]).size().rename("nombre").to_frame())

In [ ]:
if len(df):
    fig, ax = plt.subplots(figsize=(8, 3))
    for decision, group in df.groupby("décision"):
        ax.hist(group["meilleure sim."], bins=20, range=(0, 1), alpha=0.6, label=decision)
    ax.set_xlabel("similarité du meilleur passage")
    ax.set_ylabel("questions")
    ax.legend()
    plt.show()

## 3. Relire une fiche en détail

`"last"` pour la dernière, ou le début d'un identifiant de la colonne `id`.

In [ ]:
if traces:
    print(format_trace(find_trace(traces, "last")))

## 4. Le dernier essai en lot

`uv run rag-batch` écrit un tableau dans `data/batches/`. Ici, une ligne par
cas de test et une colonne par réglage : ✓ si la source attendue est bien
retrouvée et le refus correct, ✗ sinon.

In [ ]:
batches = sorted(Path("data/batches").glob("*.csv"))
if batches:
    lot = pd.read_csv(batches[-1], sep=";", encoding="utf-8-sig")
    lot["ok"] = (lot["source_ok"].fillna(True).astype(bool) & lot["refus_ok"].astype(bool))
    print(batches[-1].name)
    display(lot.pivot_table(index="cas", columns="reglage", values="ok", aggfunc="first")
               .replace({True: "✓", False: "✗"}))
else:
    print("aucun essai en lot : lancer `uv run rag-batch --search-only`")